# YGO Baseline v2 - Single Model (8-class Rarity) + Derived Foil Tasks

Approach mới theo yêu cầu:
- **Task 1**: train **1 model duy nhất** dự đoán 8 rarity classes.
- Dùng 3 input crops: **name / art / full** với 3 augmentation pipeline khác nhau.
- **Task 2/3/4**: suy ra trực tiếp từ rarity prediction theo bảng mapping.

Pipeline: `train.csv -> 3-crop dataset -> 1 tri-branch model -> rarity logits(8) -> rarity pred -> derive name_foil/art_foil/full_foil`.

In [1]:
# !pip install -q timm albumentations opencv-python-headless scikit-learn tqdm

In [2]:

import os
import random
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


In [3]:

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


DEVICE = cuda


In [4]:

TRAIN_CSV = '/kaggle/input/competitions/ygo-card-rarity-classification/dataset/train.csv'
TRAIN_IMG_DIR = '/kaggle/input/competitions/ygo-card-rarity-classification/dataset/train_images'
TEST_IMG_DIR = '/kaggle/input/competitions/ygo-card-rarity-classification/dataset/test_images'
SAMPLE_SUB = '/kaggle/input/competitions/ygo-card-rarity-classification/dataset/test.csv'
OUTPUT_DIR = 'outputs_baseline_single_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RARITY_TO_ID = {
    'Common': 1,
    'Rare': 2,
    'Super Rare': 3,
    'Ultra Rare': 4,
    'Secret Rare': 5,
    'Prismatic Secret Rare': 6,
    'Quarter Century Secret Rare': 7,
    'Starlight Rare': 8,
}
ID_TO_RARITY = {v:k for k,v in RARITY_TO_ID.items()}

# Derive Task 2/3/4 from rarity
RARITY_TO_FOIL = {
    1: (0,0,0),
    2: (1,0,0),
    3: (0,1,0),
    4: (1,1,0),
    5: (1,1,0),
    6: (1,1,0),
    7: (1,1,1),
    8: (1,1,1),
}


## 1) Crop config + 3 augmentation pipelines

In [5]:

CROP_CFG = {
    'name': dict(x1=0.06, y1=0.00, x2=0.94, y2=0.15),
    'art':  dict(x1=0.10, y1=0.15, x2=0.90, y2=0.60),
    'full': dict(x1=0.00, y1=0.00, x2=1.00, y2=1.00),
}

def crop_by_region(img, region):
    h, w = img.shape[:2]
    c = CROP_CFG[region]
    x1, y1 = int(c['x1']*w), int(c['y1']*h)
    x2, y2 = int(c['x2']*w), int(c['y2']*h)
    return img[y1:y2, x1:x2]


def get_transforms(train=True):
    # 3 transform riêng để học tín hiệu khác nhau
    name_train = A.Compose([
        A.Resize(96, 384),
        A.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.25, hue=0.03, p=0.8),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
        A.GaussNoise(std_range=(0.01, 0.03), p=0.25),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.Normalize(), ToTensorV2(),
    ])
    art_train = A.Compose([
        A.Resize(256, 256),
        A.ColorJitter(brightness=0.15, contrast=0.20, saturation=0.20, hue=0.02, p=0.8),
        A.GaussianBlur(blur_limit=(3,5), p=0.2),
        A.Sharpen(alpha=(0.1, 0.3), p=0.2),
        A.GaussNoise(std_range=(0.01, 0.03), p=0.2),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
        A.Normalize(), ToTensorV2(),
    ])
    full_train = A.Compose([
        A.Resize(384, 384),
        A.ColorJitter(brightness=0.12, contrast=0.15, saturation=0.15, hue=0.02, p=0.7),
        A.RandomGamma(gamma_limit=(85,115), p=0.4),
        A.GaussNoise(std_range=(0.01,0.02), p=0.2),
        A.Normalize(), ToTensorV2(),
    ])

    name_val = A.Compose([A.Resize(96, 384), A.Normalize(), ToTensorV2()])
    art_val = A.Compose([A.Resize(256, 256), A.Normalize(), ToTensorV2()])
    full_val = A.Compose([A.Resize(384, 384), A.Normalize(), ToTensorV2()])

    if train:
        return {'name': name_train, 'art': art_train, 'full': full_train}
    return {'name': name_val, 'art': art_val, 'full': full_val}


## 2) Tri-crop dataset

In [6]:

class CardTriCropDataset(Dataset):
    def __init__(self, df, img_dir, transforms, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, row['id'])
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        name = crop_by_region(img, 'name')
        art  = crop_by_region(img, 'art')
        full = crop_by_region(img, 'full')

        name = self.transforms['name'](image=name)['image']
        art  = self.transforms['art'](image=art)['image']
        full = self.transforms['full'](image=full)['image']

        if self.is_test:
            return name, art, full, row['id']

        y = int(row['rarity']) - 1  # 0..7 for CE
        return name, art, full, torch.tensor(y, dtype=torch.long)


## 3) Single model: 3 branches -> fusion -> 8-class logits

In [7]:

class TriCropRarityModel(nn.Module):
    def __init__(self, backbone='efficientnet_b0', pretrained=True, n_classes=8):
        super().__init__()
        self.name_encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.art_encoder  = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.full_encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')

        feat_dim = self.name_encoder.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 3, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, n_classes)
        )

    def forward(self, x_name, x_art, x_full):
        f1 = self.name_encoder(x_name)
        f2 = self.art_encoder(x_art)
        f3 = self.full_encoder(x_full)
        f = torch.cat([f1, f2, f3], dim=1)
        return self.head(f)


## 4) Train (Task1 only) + metric

In [8]:

@dataclass
class CFG:
    n_splits: int = 5
    epochs: int = 10
    batch_size: int = 12
    lr: float = 2e-4
    wd: float = 0.7e-4
    num_workers: int = 4

cfg = CFG()

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')


from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

def train_model(train_df):

    tr_df, va_df = train_test_split(
        train_df,
        test_size=0.2,
        stratify=train_df['rarity'],
        random_state=SEED
    )

    tr_ds = CardTriCropDataset(tr_df, TRAIN_IMG_DIR, get_transforms(train=True), is_test=False)
    va_ds = CardTriCropDataset(va_df, TRAIN_IMG_DIR, get_transforms(train=False), is_test=False)

    tr_dl = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True,
                       num_workers=cfg.num_workers, pin_memory=True)

    va_dl = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True)

    model = TriCropRarityModel(
        backbone='efficientnet_b0',
        pretrained=True,
        n_classes=8
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    best_f1 = -1
    best_path = os.path.join(OUTPUT_DIR, 'rarity_best.pth')

    for epoch in range(cfg.epochs):

        # ================= TRAIN =================
        model.train()
        train_loss = 0

        pbar = tqdm(tr_dl, desc=f"Epoch {epoch+1}/{cfg.epochs} [TRAIN]")

        for n,a,f,y in pbar:
            n,a,f,y = n.to(DEVICE), a.to(DEVICE), f.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(n,a,f)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * y.size(0)

            pbar.set_postfix(loss=loss.item())

        train_loss /= len(tr_dl.dataset)

        # ================= VALID =================
        model.eval()
        val_loss = 0
        pred_cls, true_cls = [], []

        pbar = tqdm(va_dl, desc=f"Epoch {epoch+1}/{cfg.epochs} [VALID]")

        with torch.no_grad():
            for n,a,f,y in pbar:

                n,a,f,y = n.to(DEVICE), a.to(DEVICE), f.to(DEVICE), y.to(DEVICE)

                logits = model(n,a,f)
                loss = criterion(logits, y)

                val_loss += loss.item() * y.size(0)

                preds = torch.argmax(logits, dim=1)

                pred_cls.extend(preds.cpu().numpy().tolist())
                true_cls.extend(y.cpu().numpy().tolist())

                pbar.set_postfix(loss=loss.item())

        val_loss /= len(va_dl.dataset)

        f1 = macro_f1(true_cls, pred_cls)

        print(
            f"\nEpoch {epoch+1}/{cfg.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_macro_f1={f1:.4f}"
        )

        # save best model
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), best_path)
            print("✅ Saved best model")

    print("\nBest F1:", best_f1)

    return best_path

In [9]:

train_df = pd.read_csv(TRAIN_CSV)

if train_df['rarity'].dtype == object:
    train_df['rarity'] = train_df['rarity'].map(RARITY_TO_ID)

model_path = train_model(train_df)

print("Saved model:", model_path)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch 1/10 [TRAIN]:   0%|          | 0/261 [00:00<?, ?it/s]

Epoch 1/10 [VALID]:   0%|          | 0/66 [00:00<?, ?it/s]


Epoch 1/10 | train_loss=0.7457 | val_loss=0.3107 | val_macro_f1=0.9010
✅ Saved best model


Epoch 2/10 [TRAIN]:   0%|          | 0/261 [00:00<?, ?it/s]

...

Epoch 9/10 [VALID]:   0%|          | 0/66 [00:00<?, ?it/s]


Epoch 9/10 | train_loss=0.0769 | val_loss=0.2162 | val_macro_f1=0.9477


Epoch 10/10 [TRAIN]:   0%|          | 0/261 [00:00<?, ?it/s]

Epoch 10/10 [VALID]:   0%|          | 0/66 [00:00<?, ?it/s]


Epoch 10/10 | train_loss=0.0875 | val_loss=0.2960 | val_macro_f1=0.8683

Best F1: 0.9532039346597383
Saved model: outputs_baseline_single_model/rarity_best.pth


## 5) Inference Task1 -> derive Task2/3/4

In [10]:

def load_model(path):
    m = TriCropRarityModel(
        backbone='efficientnet_b0',
        pretrained=False,
        n_classes=8
    ).to(DEVICE)

    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    return m


def predict_rarity(test_df, model_path):

    ds = CardTriCropDataset(test_df, TEST_IMG_DIR,
                            get_transforms(train=False),
                            is_test=True)

    dl = DataLoader(ds,
                    batch_size=cfg.batch_size,
                    shuffle=False,
                    num_workers=cfg.num_workers,
                    pin_memory=True)

    model = load_model(model_path)

    all_ids, all_pred = [], []

    with torch.no_grad():
        for n,a,f,ids in tqdm(dl, desc='predict rarity'):
            n,a,f = n.to(DEVICE), a.to(DEVICE), f.to(DEVICE)

            p = torch.softmax(model(n,a,f), dim=1).cpu().numpy()
            y = np.argmax(p, axis=1) + 1

            all_ids.extend(ids)
            all_pred.extend(y.tolist())

    return pd.DataFrame({'id': all_ids, 'rarity': all_pred})

sample = pd.read_csv(SAMPLE_SUB).copy()
test_df = sample[['id']].copy()

pred = predict_rarity(test_df, model_path)

pred['name_foil'] = pred['rarity'].map(lambda r: RARITY_TO_FOIL[int(r)][0])
pred['art_foil']  = pred['rarity'].map(lambda r: RARITY_TO_FOIL[int(r)][1])
pred['full_foil'] = pred['rarity'].map(lambda r: RARITY_TO_FOIL[int(r)][2])

sub = pred[['id','rarity','name_foil','art_foil','full_foil']].copy()

sub.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)
sub.head()

predict rarity:   0%|          | 0/140 [00:00<?, ?it/s]

,id,rarity,name_foil,art_foil,full_foil
0,222788.jpg,3,0,1,0
1,224984.jpg,4,1,1,0
2,207947.jpg,1,0,0,0
3,239644.jpg,4,1,1,0
4,480239.jpg,3,0,1,0


## 6) Notes

- Nếu muốn mạnh hơn: thay backbone `convnext_tiny` hoặc `efficientnet_b3`.
- Có thể thêm weighted CE/focal loss nếu mất cân bằng class.
- Vì task 2/3/4 là hàm xác định của rarity trong đề, cách này đảm bảo consistency 100%.